<a href="https://colab.research.google.com/github/Chunsen41/Chunsen41/blob/main/Masters_Of_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- put near your other imports ---
import time
import requests
import urllib.parse
from typing import List, Dict, Optional

# --- a session with retries + a Wikipedia-compliant User-Agent ---
HTTP_HEADERS = {
    "User-Agent": "MastersOfHomework/1.0 (https://example.com/contact) Python-requests",
    "Accept": "application/json",
}
SESSION = requests.Session()
SESSION.headers.update(HTTP_HEADERS)

def _get(url: str, *, params: Optional[Dict]=None, timeout: int = 12, retries: int = 2):
    last_err = None
    for attempt in range(retries + 1):
        try:
            resp = SESSION.get(url, params=params, timeout=timeout)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_err = e
            time.sleep(0.6 * attempt)  # simple backoff
    raise last_err


In [ ]:
def topic_from_text(text: str):
    tl = text.lower()
    if any(k in tl for k in ["laundry","detergent","washing","stain","bleach","dryer","delicates"]):
        return "Laundry"
    if any(k in tl for k in ["tutor","tutoring","study","lesson","syllabus","homework help","teacher","algebra","physics","sat","gre","test prep"]):
        return "Tutoring"
    if any(k in tl for k in ["pet","dog","puppy","cat","groom","walk","sitter","boarding","vet","vaccination"]):
        return "Pet Care"
    return None

def internet_answer(topic: str, text: str):
    preferred_pages = {
        "Laundry": ["Laundry", "Laundry detergent", "Stain removal", "Washing machine"],
        "Tutoring": ["Tutoring", "Pedagogy", "Learning theory", "Spaced repetition"],
        "Pet Care": ["Dog", "Puppy", "Dog walking", "Cat", "Animal welfare"]
    }
    titles = wiki_search_titles(text, 3)
    if not titles:
        titles = preferred_pages.get(topic, [])
    results = []
    for t in titles:
        s, u = wiki_summary(t)
        if s:
            results.append({"title": t, "snippet": s[:1000], "url": u})
        if len(results) >= 3:
            break
    return results


In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_SUMMARY = "https://en.wikipedia.org/api/rest_v1/page/summary/"

def wiki_search_titles(query: str, limit: int = 3) -> List[str]:
    """Search Wikipedia for page titles. Returns a small list of titles."""
    try:
        # origin=* keeps CORS happy; harmless server-side too
        params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": str(limit),
            "utf8": "1",
            "origin": "*",
        }
        r = _get(WIKI_API, params=params, timeout=12, retries=2)
        data = r.json()
        return [hit["title"] for hit in data.get("query", {}).get("search", [])]
    except Exception:
        return []

def wiki_summary(title: str):
    """Fetch a readable page summary + canonical URL."""
    try:
        url = WIKI_SUMMARY + urllib.parse.quote(title)
        r = _get(url, timeout=12, retries=2)
        j = r.json()
        txt = (j.get("extract") or "").strip()
        page_url = (
            j.get("content_urls", {}).get("desktop", {},).get("page")
            or j.get("content_urls", {}).get("mobile", {},).get("page")
            or f"https://en.wikipedia.org/wiki/{urllib.parse.quote(title.replace(' ', '_'))}"
        )
        return txt, page_url
    except Exception:
        return "", None


In [ ]:
# --- keep these helpers above the UI ---
def test_internet():
    try:
        r = _get("https://en.wikipedia.org/api/rest_v1/page/summary/Laundry", timeout=8, retries=1)
        j = r.json()
        return f"✅ Online. Sample title: {j.get('title','?')}"
    except Exception as e:
        return f"❌ No internet or blocked. Error: {e}"


In [ ]:
def build_app():
    with gr.Blocks(title=APP_TITLE, theme=Brand, css=CSS) as demo:
        # ... navbar, hero, other tabs ...

        with gr.Tabs():
            # ... other tabs ...

            with gr.Tab("Assistant (Chatbot)", elem_classes=["card"]):
                gr.Markdown("Ask about **Laundry**, **Tutoring**, or **Pet Care** — I’ll fetch live info and show local stats.")

                chatbot = gr.Chatbot(value=[], height=380, type="tuples", render_markdown=True, elem_classes=["chatbox"])
                chat_in  = gr.Textbox(placeholder="e.g., Best way to remove grease stains? | Study tips for Algebra 2 | How often should I walk a puppy?")
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear")

                # ⬇️ Add the internet test UI **inside** the Blocks context
                with gr.Row():
                    test_btn = gr.Button("Test internet")
                    test_out = gr.Markdown()

                # wire events INSIDE Blocks
                send_btn.click(chatbot_reply, [chat_in, chatbot], chatbot)
                send_btn.click(lambda: "", None, chat_in)
                clear_btn.click(lambda: [], None, chatbot)

                # ✅ correct binding location:
                test_btn.click(lambda: test_internet(), [], test_out)

        # footer...
    return demo

# bootstrap
if __name__ == "__main__":
    init_db()
    app = build_app()   # make sure you build inside a function or a single Blocks context
    app.launch()


/tmp/ipython-input-745259978.py:11: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  chatbot = gr.Chatbot(value=[], height=380, type="tuples", render_markdown=True, elem_classes=["chatbox"])


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a17bb71c7b75241b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# app.py (Part 1/2) — Core app + DB + Modern "hero curve" styling
# Adds live internet answers for Laundry, Tutoring, and Pet Care via Wikipedia
# Title: Masters of Homework (kept as requested)

import os, re, sqlite3, hashlib, datetime, statistics
import gradio as gr
import requests, urllib.parse  # for internet answers

APP_TITLE = "Masters of Homework"
DB_PATH = os.environ.get("MARKET_DB", "marketplace.db")

CATEGORIES = ["Laundry", "Tutoring", "Pet Care"]  # <- three verticals

# ---- Theme + CSS (navbar + curved hero) ----
Brand = gr.themes.Base(
    primary_hue="sky",
    secondary_hue="slate",
    neutral_hue="slate",
    text_size="sm",
    spacing_size="sm",
    radius_size="lg",
).set(
    body_background_fill="linear-gradient(120deg, #0ea5e9 0%, #0b1020 62%, #0b1220 100%)",
    block_background_fill="#ffffff",
    block_shadow="0 10px 28px rgba(0,0,0,.18)",
    button_primary_background_fill="#0ea5e9",
    button_primary_background_fill_hover="#0284c7",
    input_background_fill="#f8fafc",
)

CSS = """
*{box-sizing:border-box}
body, input, textarea, button, select{
  font-family: Inter, ui-sans-serif, -apple-system, Segoe UI, Roboto, sans-serif !important;
  color:#0f172a;
}

/* NAVBAR */
.navbar {
  display:flex; gap:18px; align-items:center; justify-content:space-between;
  padding:14px 22px; max-width:1100px; margin:0 auto; color:#e5e7eb;
}
.navbar .brand {font-weight:800; letter-spacing:.4px}
.navbar .links a {color:#e5e7eb; text-decoration:none; margin:0 10px; opacity:.9}
.navbar .links a:hover {opacity:1; text-decoration:underline}

/* HERO with curved bottom */
.hero-wrap{ position:relative; overflow:hidden; border-radius:22px; max-width:1100px; margin:14px auto 28px; }
.hero {
  background: url('https://images.unsplash.com/photo-1485727749690-d091e8284ef5?q=80&w=1600&auto=format&fit=crop') center/cover no-repeat,
              linear-gradient(180deg, rgba(2,6,23,.85), rgba(2,6,23,.85));
  min-height: 320px; color:#fff; display:flex; align-items:center; justify-content:center; text-align:center;
  padding: 36px 24px;
}
.hero h1 { font-size:38px; margin:6px 0 8px }
.hero p { opacity:.92; max-width:760px; margin:0 auto 18px }
.hero .cta { padding:10px 16px; border-radius:999px; background:#0ea5e9; color:#03121e; display:inline-block; font-weight:700 }
.hero .cta:hover{ filter:brightness(.95) }

/* Curve effect */
.hero-wrap:after{
  content:""; position:absolute; left:50%; transform:translateX(-50%);
  bottom:-50px; width:160%; height:120px; background:#fff; border-radius:0 0 50% 50%;
  box-shadow:0 18px 40px rgba(2,6,23,.25);
}

/* CARDS */
.card{ background:#fff; border:1px solid rgba(15,23,42,.06); border-radius:18px; padding:16px;
  box-shadow:0 8px 26px rgba(2,6,23,.12); backdrop-filter:blur(8px); }
.badge{display:inline-block;padding:4px 10px;border-radius:999px;font-size:12px;margin-left:6px}
.badge--verified{background:#dcfce7;color:#166534}
.price{font-weight:700;color:#22c55e}

/* CHAT */
.chatbox{max-width:1100px;margin:10px auto}
"""

# ---- DB helpers & schema (includes reviews/offers etc.) ----
def db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    schema = """
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS users(
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      role TEXT CHECK(role IN ('user','provider')) NOT NULL,
      name TEXT NOT NULL,
      email TEXT UNIQUE NOT NULL,
      password_hash TEXT NOT NULL,
      city TEXT DEFAULT '',
      phone TEXT DEFAULT '',
      created_at TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS provider_services(
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      provider_id INTEGER NOT NULL,
      category TEXT NOT NULL,
      bio TEXT DEFAULT '',
      base_price REAL DEFAULT 0,
      is_verified INTEGER DEFAULT 0,
      FOREIGN KEY(provider_id) REFERENCES users(id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS jobs(
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      user_id INTEGER NOT NULL,
      category TEXT NOT NULL,
      description TEXT NOT NULL,
      city TEXT NOT NULL,
      preferred_time TEXT NOT NULL,
      status TEXT CHECK(status IN ('open','accepted','in_progress','completed','cancelled')) DEFAULT 'open',
      created_at TEXT NOT NULL,
      FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS job_offers(
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      job_id INTEGER NOT NULL,
      provider_id INTEGER NOT NULL,
      message TEXT DEFAULT '',
      offer_price REAL DEFAULT 0,
      status TEXT CHECK(status IN ('pending','accepted','rejected')) DEFAULT 'pending',
      created_at TEXT NOT NULL,
      FOREIGN KEY(job_id) REFERENCES jobs(id) ON DELETE CASCADE,
      FOREIGN KEY(provider_id) REFERENCES users(id) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS reviews(
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      job_id INTEGER NOT NULL,
      user_id INTEGER NOT NULL,
      provider_id INTEGER NOT NULL,
      rating INTEGER CHECK(rating BETWEEN 1 AND 5) NOT NULL,
      comment TEXT DEFAULT '',
      created_at TEXT NOT NULL,
      FOREIGN KEY(job_id) REFERENCES jobs(id) ON DELETE CASCADE,
      FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE CASCADE,
      FOREIGN KEY(provider_id) REFERENCES users(id) ON DELETE CASCADE
    );
    """
    conn = db(); conn.executescript(schema); conn.commit(); conn.close()

def hash_pw(pw: str) -> str:
    return hashlib.sha256(pw.encode("utf-8")).hexdigest()

# ---- Auth ----
def signup(role, name, email, password, city, phone):
    if not all([role, name, email, password]):
        return "All fields required", None
    try:
        conn = db(); cur = conn.cursor()
        cur.execute("""INSERT INTO users(role,name,email,password_hash,city,phone,created_at)
                       VALUES (?,?,?,?,?,?,?)""",
                    (role, name.strip(), email.lower().strip(), hash_pw(password),
                     (city or "").strip(), (phone or "").strip(), datetime.datetime.utcnow().isoformat()))
        conn.commit()
        uid = cur.lastrowid
        return f"✅ Signed up as {role}. You can log in now.", {"id": uid, "role": role, "name": name, "email": email}
    except sqlite3.IntegrityError:
        return "❌ Email already exists.", None
    finally:
        conn.close()

def login(email, password):
    conn = db(); cur = conn.cursor()
    cur.execute("SELECT * FROM users WHERE email=?", (email.lower().strip(),))
    row = cur.fetchone(); conn.close()
    if not row or row["password_hash"] != hash_pw(password):
        return "❌ Invalid credentials.", None
    return f"👋 Welcome back, {row['name']}!", {"id": row["id"], "role": row["role"], "name": row["name"], "email": row["email"]}

# ---- Provider ----
def upsert_provider_service(user, category, bio, base_price, is_verified):
    if not user or user.get("role") != "provider":
        return "❌ Must be logged in as provider."
    conn = db(); cur = conn.cursor()
    cur.execute("SELECT id FROM provider_services WHERE provider_id=? AND category=?", (user["id"], category))
    existing = cur.fetchone()
    if existing:
        cur.execute("UPDATE provider_services SET bio=?, base_price=?, is_verified=? WHERE id=?",
                    (bio, float(base_price or 0), 1 if is_verified else 0, existing["id"]))
    else:
        cur.execute("""INSERT INTO provider_services(provider_id,category,bio,base_price,is_verified)
                       VALUES (?,?,?,?,?)""",
                    (user["id"], category, bio, float(base_price or 0), 1 if is_verified else 0))
    conn.commit(); conn.close()
    return "✅ Service saved."

def list_open_jobs_for_provider(user, city_filter, category_filter):
    if not user or user.get("role") != "provider":
        return "❌ Please log in as provider."
    conn = db(); cur = conn.cursor()
    query = ("SELECT j.*, u.name AS requester FROM jobs j JOIN users u ON j.user_id=u.id WHERE j.status='open'")
    params=[]
    if city_filter:
        query += " AND j.city LIKE ?"; params.append(f"%{city_filter.strip()}%")
    if category_filter and category_filter != "All":
        query += " AND j.category=?"; params.append(category_filter)
    query += " ORDER BY j.created_at DESC LIMIT 50"
    cur.execute(query, params); rows = cur.fetchall(); conn.close()
    if not rows: return "No open jobs match your filters."
    return "\n\n".join([f"#{r['id']} | {r['category']} | {r['city']} | when: {r['preferred_time']} | by: {r['requester']}\n{r['description']}" for r in rows])

def submit_offer(user, job_id, offer_price, message):
    if not user or user.get("role") != "provider": return "❌ Must be logged in as provider."
    conn = db(); cur = conn.cursor()
    cur.execute("SELECT id,status FROM jobs WHERE id=?", (job_id,)); job=cur.fetchone()
    if not job: conn.close(); return "❌ Job not found."
    if job["status"]!="open": conn.close(); return "⚠️ Job is not open."
    cur.execute("""INSERT INTO job_offers(job_id,provider_id,message,offer_price,created_at)
                   VALUES (?,?,?,?,?)""",
                (job_id, user["id"], message or "", float(offer_price or 0), datetime.datetime.utcnow().isoformat()))
    conn.commit(); conn.close(); return "✅ Offer sent!"

# ---- User ----
def create_job(user, category, description, city, preferred_time):
    if not user or user.get("role") != "user": return "❌ Must be logged in as user."
    if not all([category, description, city, preferred_time]): return "❌ Please fill all fields."
    conn = db(); cur = conn.cursor()
    cur.execute("""INSERT INTO jobs(user_id,category,description,city,preferred_time,status,created_at)
                   VALUES (?,?,?,?,?,'open',?)""",
                (user["id"], category, description.strip(), city.strip(), preferred_time.strip(), datetime.datetime.utcnow().isoformat()))
    conn.commit(); jid = cur.lastrowid; conn.close()
    return f"✅ Request #{jid} created! Providers can now send offers."

def my_jobs(user):
    if not user or user.get("role")!="user": return "❌ Please log in as user."
    conn=db(); cur=conn.cursor()
    cur.execute("""SELECT j.*, (SELECT COUNT(*) FROM job_offers o WHERE o.job_id=j.id) AS offers
                   FROM jobs j WHERE j.user_id=? ORDER BY j.created_at DESC""", (user["id"],))
    rows=cur.fetchall(); conn.close()
    if not rows: return "No requests yet."
    return "\n\n".join([f"#{r['id']} [{r['status']}] {r['category']} | {r['city']} | {r['preferred_time']} | offers: {r['offers']}\n{r['description']}" for r in rows])

def view_offers(user, job_id):
    if not user or user.get("role")!="user": return "❌ Please log in as user."
    conn=db(); cur=conn.cursor()
    cur.execute("""SELECT o.id, o.offer_price, o.message, o.status, u.name AS provider_name
                   FROM job_offers o JOIN users u ON o.provider_id=u.id
                   WHERE o.job_id=? ORDER BY o.created_at DESC""", (job_id,))
    rows=cur.fetchall(); conn.close()
    if not rows: return "No offers yet."
    return "\n\n".join([f"Offer #{r['id']} from {r['provider_name']} | ${r['offer_price']:.2f} | {r['status']}\n{r['message']}" for r in rows])

def accept_offer(user, offer_id):
    if not user or user.get("role")!="user": return "❌ Must be logged in as user."
    conn=db(); cur=conn.cursor()
    cur.execute("""SELECT o.id, o.job_id, j.user_id, j.status
                   FROM job_offers o JOIN jobs j ON o.job_id=j.id WHERE o.id=?""",(offer_id,))
    row=cur.fetchone()
    if not row: conn.close(); return "❌ Offer not found."
    if row["user_id"]!=user["id"]: conn.close(); return "❌ This request is not yours."
    if row["status"] not in ("open","accepted"): conn.close(); return "⚠️ Cannot accept now."
    cur.execute("UPDATE job_offers SET status='accepted' WHERE id=?", (offer_id,))
    cur.execute("UPDATE job_offers SET status='rejected' WHERE job_id=? AND id<>?", (row["job_id"], offer_id))
    cur.execute("UPDATE jobs SET status='accepted' WHERE id=?", (row["job_id"],))
    conn.commit(); conn.close(); return "✅ Offer accepted."

def update_job_status(user, job_id, new_status):
    if not user: return "❌ Login required."
    conn=db(); cur=conn.cursor()
    cur.execute("SELECT * FROM jobs WHERE id=?", (job_id,)); job=cur.fetchone()
    if not job: conn.close(); return "❌ Job not found."
    allowed=False
    if user["role"]=="user" and job["user_id"]==user["id"]: allowed=True
    elif user["role"]=="provider":
        cur.execute("""SELECT 1 FROM job_offers WHERE job_id=? AND provider_id=? AND status='accepted'""",(job_id, user["id"]))
        allowed = cur.fetchone() is not None
    if not allowed: conn.close(); return "❌ Not allowed."
    if new_status not in ["in_progress","completed","cancelled"]: conn.close(); return "❌ Invalid status."
    cur.execute("UPDATE jobs SET status=? WHERE id=?", (new_status, job_id)); conn.commit(); conn.close()
    return f"✅ Status updated to {new_status}."

def leave_review(user, job_id, rating, comment):
    if not user or user.get("role")!="user": return "❌ Must be logged in as user."
    rating=int(rating)
    conn=db(); cur=conn.cursor()
    cur.execute("""SELECT o.provider_id, j.status FROM job_offers o JOIN jobs j ON o.job_id=j.id
                   WHERE o.job_id=? AND o.status='accepted'""",(job_id,))
    row=cur.fetchone()
    if not row: conn.close(); return "❌ No accepted provider."
    if row["status"]!="completed": conn.close(); return "⚠️ Only completed requests can be reviewed."
    cur.execute("""INSERT INTO reviews(job_id,user_id,provider_id,rating,comment,created_at)
                   VALUES (?,?,?,?,?,?)""",
                (job_id, user["id"], row["provider_id"], rating, comment or "", datetime.datetime.utcnow().isoformat()))
    conn.commit(); conn.close(); return "⭐ Thanks for your review!"

def list_providers(category, city_like):
    conn=db(); cur=conn.cursor()
    q=("SELECT u.name, u.city, u.phone, s.category, s.base_price, s.bio, s.is_verified "
       "FROM provider_services s JOIN users u ON s.provider_id=u.id WHERE 1=1")
    params=[]
    if category and category!="All": q+=" AND s.category=?"; params.append(category)
    if city_like: q+=" AND u.city LIKE ?"; params.append(f"%{city_like.strip()}%")
    cur.execute(q, params); rows=cur.fetchall(); conn.close()
    if not rows: return "No providers yet."
    out=[]
    for r in rows:
        badge = "<span class='badge badge--verified'>Verified</span>" if r["is_verified"] else "—"
        out.append(f"**{r['name']}** ({r['city']}) {badge}<br>"
                   f"<span class='price'>${r['base_price']:.2f}</span> · {r['category']}<br>"
                   f"{r['bio']} · 📞 {r['phone'] or 'N/A'}")
    return "\n\n".join(out)

# ---- Tool for stats (used by bot) ----
def get_category_stats(category: str|None):
    if not category: return None
    conn=db(); cur=conn.cursor()
    cur.execute("""SELECT u.city, s.base_price FROM provider_services s JOIN users u ON s.provider_id=u.id
                   WHERE s.category=?""",(category,))
    rows=cur.fetchall(); conn.close()
    if not rows: return {"count":0,"avg_price":None,"top_cities":[]}
    prices=[r[1] for r in rows if r[1] is not None]
    avg = round(statistics.mean(prices),2) if prices else None
    city_counts={}
    for r in rows:
        c=r[0] or "Unknown"; city_counts[c]=city_counts.get(c,0)+1
    top=sorted(city_counts.items(), key=lambda x:x[1], reverse=True)[:3]
    return {"count":len(rows), "avg_price":avg, "top_cities":top}


In [ ]:
# app.py (Part 2/2) — Internet Q&A for Laundry/Tutoring/Pet + Chatbot + UI

# ---- Internet helpers (Wikipedia) for all three verticals ----
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_SUMMARY = "https://en.wikipedia.org/api/rest_v1/page/summary/"

def wiki_search_titles(query: str, limit: int = 3):
    try:
        r = requests.get(WIKI_API, params={
            "action":"query","list":"search","srsearch":query,"format":"json","srlimit":str(limit),"utf8":"1"
        }, timeout=10)
        r.raise_for_status(); data=r.json()
        return [hit["title"] for hit in data.get("query",{}).get("search",[])]
    except Exception:
        return []

def wiki_summary(title: str):
    try:
        url = WIKI_SUMMARY + urllib.parse.quote(title)
        r = requests.get(url, timeout=10, headers={"accept":"application/json"})
        r.raise_for_status(); j=r.json()
        txt = (j.get("extract") or "").strip()
        page_url = j.get("content_urls",{}).get("desktop",{}).get("page") or j.get("content_urls",{}).get("mobile",{}).get("page")
        return txt, page_url or f"https://en.wikipedia.org/wiki/{urllib.parse.quote(title.replace(' ','_'))}"
    except Exception:
        return "", None

def topic_from_text(text: str):
    tl=text.lower()
    if any(k in tl for k in ["laundry","detergent","washing","stain","bleach","dryer","delicates"]):
        return "Laundry"
    if any(k in tl for k in ["tutor","tutoring","study","lesson","syllabus","homework help","teacher"]):
        return "Tutoring"
    if any(k in tl for k in ["pet","dog","cat","groom","walk","sitter","boarding","vet"]):
        return "Pet Care"
    return None

def internet_answer(topic: str, text: str):
    # map some intents to better pages
    preferred_pages = {
        "Laundry": ["Laundry", "Laundry detergent", "Stain removal", "Washing machine"],
        "Tutoring": ["Tutoring", "Pedagogy", "Learning theory", "Spaced repetition"],
        "Pet Care": ["Dog walking", "Cat", "Dog", "Animal care and service workers"]
    }
    titles = wiki_search_titles(text, 3)
    results=[]
    for t in (titles or preferred_pages.get(topic, [])):
        s,u = wiki_summary(t)
        if s:
            results.append({"title":t,"snippet":s[:1000],"url":u})
        if len(results)>=3: break
    return results

# ---- Chatbot (uses internet for the three topics; else marketplace RAG/tools baseline) ----
def pretty_top_cities(top):
    if not top: return "No city data yet."
    return ", ".join([f"{c} ({n})" for c,n in top])

def chatbot_reply(message: str, history: list[tuple[str,str]]|None):
    if history is None: history=[]
    text=(message or "").strip()
    if not text:
        return history + [("", "Ask about Laundry, Tutoring, or Pet Care — pricing, tips, or general info!")]

    topic = topic_from_text(text)
    if topic:
        hits = internet_answer(topic, text)
        stats = get_category_stats(topic)
        section_stats = ""
        if stats:
            avg = f"${stats['avg_price']:.2f}" if stats["avg_price"] is not None else "N/A"
            section_stats = (f"\n**Local stats ({topic})**  \n"
                             f"- Providers: **{stats['count']}**  \n"
                             f"- Avg base price: **{avg}**  \n"
                             f"- Top cities: **{pretty_top_cities(stats['top_cities'])}**\n")
        if hits:
            bullets = "\n\n".join([f"- **{h['title']}** — {h['snippet']}  \n  *Source:* [{h['url']}]({h['url']})" for h in hits])
            bot = f"### {topic} — live info\n{bullets}\n{section_stats}\n> Tip: Always follow product labels and local regulations."
            return history + [(text, bot)]
        else:
            bot = f"I tried fetching live info on **{topic}** but couldn't reach sources just now.{section_stats}"
            return history + [(text, bot)]

    # Fallback to marketplace guidance (no external web)
    bot = ("I can help match you with providers. Use the tabs to **Post a Request**, "
           "**List Open Jobs** (for providers), or **Browse Providers**.")
    return history + [(text, bot)]

# ---- Build the UI (navbar + hero + tabs) ----
def build_app():
    with gr.Blocks(title=APP_TITLE, theme=Brand, css=CSS) as demo:
        # Navbar
        gr.HTML(
            "<div class='navbar'>"
            f"<div class='brand'>🧠 {APP_TITLE}</div>"
            "<div class='links'>"
            "<a href='#'>Home</a><a href='#'>Services</a><a href='#'>How it works</a><a href='#'>Contact</a>"
            "</div></div>"
        )
        # Hero (curved)
        gr.HTML(
            "<div class='hero-wrap'>"
            "<div class='hero'>"
            "<div>"
            "<h1>Welcome to Masters of Homework</h1>"
            "<p>On-demand marketplace for <b>Laundry</b>, <b>Tutoring</b>, and <b>Pet Care</b>. "
            "Post requests, compare offers, and get things done.</p>"
            "<a class='cta' href='#'>Get Started</a>"
            "</div></div></div>"
        )

        session = gr.State(value=None)

        # Auth
        with gr.Row():
            with gr.Column(elem_classes=["card"]):
                gr.Markdown("### Create an account")
                role = gr.Radio(["user","provider"], label="Role", value="user")
                name = gr.Textbox(label="Name"); email = gr.Textbox(label="Email")
                password = gr.Textbox(label="Password", type="password")
                city = gr.Textbox(label="City"); phone = gr.Textbox(label="Phone (optional)")
                signup_btn = gr.Button("Sign up", variant="primary"); signup_out = gr.Markdown()
            with gr.Column(elem_classes=["card"]):
                gr.Markdown("### Log in")
                le = gr.Textbox(label="Email"); lp = gr.Textbox(label="Password", type="password")
                login_btn = gr.Button("Log in"); login_out = gr.Markdown()

        signup_btn.click(lambda r,n,e,p,c,ph: signup(r,n,e,p,c,ph),
                         [role,name,email,password,city,phone],[signup_out, session])
        login_btn.click(lambda e,p: login(e,p), [le,lp], [login_out, session])

        with gr.Tabs():
            with gr.Tab("User: Post & Manage", elem_classes=["card"]):
                with gr.Row():
                    u_cat = gr.Dropdown(CATEGORIES, label="Category")
                    u_desc = gr.Textbox(label="Describe your need", lines=3)
                    u_city = gr.Textbox(label="City")
                    u_time = gr.Textbox(label="Preferred time (YYYY-MM-DD HH:MM)")
                post = gr.Button("Post Request", variant="primary"); post_out = gr.Markdown()

                my_btn = gr.Button("Refresh My Requests"); my_out = gr.Markdown()

                with gr.Row():
                    jid = gr.Number(label="Request ID to view offers", precision=0)
                    view_o = gr.Button("View Offers")
                offers_out = gr.Markdown()

                with gr.Row():
                    oid = gr.Number(label="Offer ID to accept", precision=0)
                    accept_btn = gr.Button("Accept Offer")
                accept_out = gr.Markdown()

                with gr.Row():
                    ujid = gr.Number(label="Request ID", precision=0)
                    new_status = gr.Dropdown(["in_progress","completed","cancelled"], label="Update Status")
                    upd_btn = gr.Button("Update")
                upd_out = gr.Markdown()

                with gr.Row():
                    rjid = gr.Number(label="Completed Request ID", precision=0)
                    rr = gr.Slider(1,5,step=1,label="Rating"); rc = gr.Textbox(label="Comment", lines=2)
                    review_btn = gr.Button("Leave Review")
                review_out = gr.Markdown()

                post.click(create_job, [session,u_cat,u_desc,u_city,u_time], post_out)
                my_btn.click(my_jobs, [session], my_out)
                view_o.click(view_offers, [session, jid], offers_out)
                accept_btn.click(accept_offer, [session, oid], accept_out)
                upd_btn.click(update_job_status, [session, ujid, new_status], upd_out)
                review_btn.click(leave_review, [session, rjid, rr, rc], review_out)

            with gr.Tab("Provider: Services & Offers", elem_classes=["card"]):
                ps_cat = gr.Dropdown(CATEGORIES, label="Category")
                ps_bio = gr.Textbox(label="Short Bio", lines=3)
                ps_price = gr.Number(label="Base price")
                ps_ver = gr.Checkbox(label="I'm verified")
                ps_save = gr.Button("Save/Update", variant="primary"); ps_out = gr.Markdown()

                with gr.Row():
                    pf_city = gr.Textbox(label="Filter: city contains")
                    pf_cat = gr.Dropdown(["All"]+CATEGORIES, label="Filter: category", value="All")
                    prov_refresh = gr.Button("List Open Requests")
                prov_jobs_out = gr.Markdown()

                with gr.Row():
                    bid_job_id = gr.Number(label="Request ID", precision=0)
                    bid_price = gr.Number(label="Offer price")
                    bid_msg = gr.Textbox(label="Message", lines=2)
                    bid_btn = gr.Button("Send Offer")
                bid_out = gr.Markdown()

                ps_save.click(upsert_provider_service, [session, ps_cat, ps_bio, ps_price, ps_ver], ps_out)
                prov_refresh.click(list_open_jobs_for_provider, [session, pf_city, pf_cat], prov_jobs_out)
                bid_btn.click(submit_offer, [session, bid_job_id, bid_price, bid_msg], bid_out)

            with gr.Tab("Browse Providers", elem_classes=["card"]):
                b_cat = gr.Dropdown(["All"]+CATEGORIES, label="Category", value="All")
                b_city = gr.Textbox(label="City contains")
                b_btn = gr.Button("Search", variant="primary")
                b_out = gr.Markdown()
                b_btn.click(list_providers, [b_cat, b_city], b_out)

            with gr.Tab("Assistant (Chatbot)", elem_classes=["card"]):
                gr.Markdown("Ask about **Laundry**, **Tutoring**, or **Pet Care** — I’ll fetch live info and show local stats.")
                chat = gr.Chatbot(value=[], height=380, type="tuples", render_markdown=True, elem_classes=["chatbox"])
                chat_in = gr.Textbox(placeholder="e.g., Best way to remove grease stains?  |  Study tips for Algebra 2  |  How often should I walk a puppy?")
                send = gr.Button("Send", variant="primary"); clear = gr.Button("Clear")
                send.click(chatbot_reply, [chat_in, chat], chat)
                send.click(lambda:"", None, chat_in)
                clear.click(lambda: [], None, chat)

        gr.Markdown("<div class='card' style='max-width:1100px;margin:16px auto'>"
                    "<b>Note:</b> Demo MVP. Add payments, verification, and moderation for production."
                    "</div>")
    return demo

# ---- Boot ----
if __name__ == "__main__":
    init_db()
    app = build_app()
    app.launch()


/tmp/ipython-input-2977169263.py:205: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  chat = gr.Chatbot(value=[], height=380, type="tuples", render_markdown=True, elem_classes=["chatbox"])


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40b48867c732d07860.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
